# Cooperative & Competitive RL Training

Trains two MaskablePPO agents for the Peer-Simulation labor market.

**Runtime**: GPU (T4 or better). Full 3M steps takes ~1-2 hours.

**Steps:**
1. Run **Setup** - installs packages and writes source files
2. Run **Train Cooperative** - saves `coop_model_longrun.zip` + vecnorm
3. Run **Train Competitive** - saves `comp_model_longrun.zip` + vecnorm
4. Run **Download** - zips and downloads all 4 output files

> After training, copy the 4 files into `cooperative/` and `competitive/` in your local repo.

## 1 - Setup

In [ ]:
# Check GPU
import subprocess, sys
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'No GPU detected - training on CPU')

# Install packages matching local environment
!pip install -q mesa==3.0.3 stable-baselines3==2.7.1 sb3-contrib==2.7.1


In [ ]:
import base64, pathlib, os

os.makedirs('/content/cooperative', exist_ok=True)
os.makedirs('/content/competitive', exist_ok=True)

BLOBS = {"model_rl": "...", "coop_env": "IyBjb29wZXJhdGl2ZS9maXJtX2Vudi5weQojCiMgTXVsdGktYWdlbnQgZW52OiBOX1JMX0ZJUk1TIGZpcm1zIHNoYXJlIHRoZSBzYW1lIHNpbXVsYXRpb24uCiMgQ09PUEVSQVRJVkUgbW9kZTogcmV3YXJkID0gYXZlcmFnZSBwcm9maXQgb2YgYWxsIFJMIGZpcm1zLgojIFVzZXMgcm91bmQtcm9iaW46IGVhY2ggZ3ltLnN0ZXAoKSBhY3RzIGZvciBvbmUgUkwgZmlybSBpbiB0dXJuLgojCiMgT2JzZXJ2YXRpb24gKDEzIGZlYXR1cmVzKToKIyAgIDAgIHByb2ZpdF9zaWduYWwgICAgICAg4oCUIHRhbmgocHJvZml0LzVrKSAgICAgICAgICAgICBvd24gcHJvZml0IGxldmVsCiMgICAxICBwcm9maXRfY2hhbmdlICAgICAgIOKAlCB0YW5oKM6UcHJvZml0LzJrKSAgICAgICAgICAgIG93biBwcm9maXQgdHJlbmQKIyAgIDIgIHZtcGxfZ2FwICAgICAgICAgICAg4oCUIHRhbmgoKFZNUEwtd2FnZSkvd2FnZSkgICAgICAgb3Zlci91bmRlcnBheWluZyB2cyBNUEwKIyAgIDMgIHdhZ2VfdnNfbWt0ICAgICAgICAg4oCUIHRhbmgoKG93bi1ta3QpL21rdCkgICAgICAgICB3YWdlIHZzIEFMTCBmaXJtcycgYXZnCiMgICA0ICBsYWJvcl9yYXRpbyAgICAgICAgIOKAlCBuX3dvcmtlcnMvNDAgICAgICAgICAgICAgICAgd29ya2ZvcmNlIHNpemUKIyAgIDUgIHZhY2FuY3lfcmF0aW8gICAgICAg4oCUIG1pbih2YWMsNSkvNSAgICAgICAgICAgICAgICBvcGVuIHNsb3RzCiMgICA2ICB3b3JrZXJfY2hhbmdlICAgICAgIOKAlCB0YW5oKM6Ud29ya2Vycy8zKSAgICAgICAgICAgIHJlY2VudCBoZWFkY291bnQgzpQKIyAgIDcgIHdhZ2VfY2xvY2sgICAgICAgICAg4oCUIHN0ZXAlMTIvMTEgICAgICAgICAgICAgICAgICBwb3NpdGlvbiBpbiB3YWdlIGN5Y2xlCiMgICA4ICBwcm9kX3ZzX21rdCAgICAgICAgIOKAlCB0YW5oKChBLWF2Z0EpL2F2Z0EpICAgICAgICAgb3duIHByb2R1Y3Rpdml0eSB2cyBwZWVycwojICAgOSAgY2FwX3ZzX21rdCAgICAgICAgICDigJQgdGFuaCgoSy1hdmdLKS9hdmdLKSAgICAgICAgIG93biBjYXBpdGFsIHZzIHBlZXJzCiMgIDEwICBzdXJ2aXZhbF9zaWduYWwgICAgIOKAlCB0YW5oKGRlZmljaXRfbW9udGhzLzEyKSAgICAgb3duIHN1cnZpdmFsIHVyZ2VuY3kKIyAgMTEgIHRlYW1fcHJvZml0X3NpZ25hbCAg4oCUIHRhbmgobWVhbihSTCBwcm9maXRzKS81aykgICBjb2xsZWN0aXZlIHRlYW0gaGVhbHRoCiMgIDEyICBhdF9yaXNrX2ZyYWN0aW9uICAgIOKAlCB3b3JrZXJzIHcvIG1vbnRoc19iZWxvd19ta3Q+PXBhdGllbmNlLy8yIC8gbWF4KGxhYm9yLDEpCgppbXBvcnQgZ3ltbmFzaXVtIGFzIGd5bQppbXBvcnQgbnVtcHkgYXMgbnAKZnJvbSBtb2RlbF9ybCBpbXBvcnQgTGFib3JNYXJrZXRNb2RlbAoKTl9STF9GSVJNUyA9IDMKCgpjbGFzcyBDb29wRmlybUVudihneW0uRW52KToKCiAgICBkZWYgX19pbml0X18oc2VsZik6CiAgICAgICAgc2VsZi5tb2RlbCAgICAgICAgPSBMYWJvck1hcmtldE1vZGVsKG5fcmxfZmlybXM9Tl9STF9GSVJNUywgZXF1YWxfdGVybXM9VHJ1ZSkKICAgICAgICBzZWxmLnJsX2Zpcm1zICAgICA9IHNlbGYubW9kZWwuZmlybXNbOk5fUkxfRklSTVNdCiAgICAgICAgc2VsZi5jdXJyZW50X2lkeCAgPSAwCiAgICAgICAgc2VsZi5jdXJyZW50X3N0ZXAgPSAwCiAgICAgICAgc2VsZi5tYXhfc3RlcCAgICAgPSAzNjAKCiAgICAgICAgc2VsZi5wcmV2X3Byb2ZpdCAgPSB7Zi51aWQ6IDAuMCAgICAgICAgICAgICAgICAgICAgZm9yIGYgaW4gc2VsZi5ybF9maXJtc30KICAgICAgICBzZWxmLnByZXZfd29ya2VycyA9IHtmLnVpZDogbGVuKGYuY3VycmVudF93b3JrZXJzKSBmb3IgZiBpbiBzZWxmLnJsX2Zpcm1zfQoKICAgICAgICBzZWxmLmFjdGlvbl9zcGFjZSA9IGd5bS5zcGFjZXMuRGlzY3JldGUoOCkKICAgICAgICBzZWxmLm9ic2VydmF0aW9uX3NwYWNlID0gZ3ltLnNwYWNlcy5Cb3goCiAgICAgICAgICAgIGxvdz0tMS41LCBoaWdoPTEuNSwgc2hhcGU9KDEzLCksIGR0eXBlPW5wLmZsb2F0MzIKICAgICAgICApCgogICAgIyDilIDilIAgT2JzZXJ2YXRpb24gZm9yIG9uZSBSTCBmaXJtIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAoKICAgIGRlZiBfb2JzZXJ2ZShzZWxmLCBpZHgpOgogICAgICAgIGZpcm0gID0gc2VsZi5ybF9maXJtc1tpZHhdCiAgICAgICAgbW9kZWwgPSBzZWxmLm1vZGVsCgogICAgICAgIHByb2ZpdF9zaWduYWwgICAgICAgID0gZmxvYXQobnAudGFuaChmaXJtLnByb2ZpdCAvIDVfMDAwKSkKICAgICAgICBwcm9maXRfY2hhbmdlX3NpZ25hbCA9IGZsb2F0KG5wLnRhbmgoCiAgICAgICAgICAgIChmaXJtLnByb2ZpdCAtIHNlbGYucHJldl9wcm9maXRbZmlybS51aWRdKSAvIDJfMDAwCiAgICAgICAgKSkKCiAgICAgICAgbGFib3IgPSBsZW4oZmlybS5jdXJyZW50X3dvcmtlcnMpCiAgICAgICAgaWYgbGFib3IgPiAwOgogICAgICAgICAgICBtcGwgICAgICA9IGZpcm0ubWFyZ2luYWxfcHJvZHVjdF9sYWJvcihmaXJtLnByb2R1Y3Rpdml0eSwgbGFib3IsIGZpcm0uYWxwaGEpCiAgICAgICAgICAgIHZtcGwgICAgID0gbXBsICogZmlybS5vdXRwdXRfcHJpY2UKICAgICAgICAgICAgdm1wbF9nYXAgPSBmbG9hdChucC50YW5oKCh2bXBsIC0gZmlybS5tb250aGx5X3dhZ2UpIC8gbWF4KGZpcm0ubW9udGhseV93YWdlLCAxLjApKSkKICAgICAgICBlbHNlOgogICAgICAgICAgICB2bXBsX2dhcCA9IDEuMAoKICAgICAgICBhbGxfd2FnZXMgICA9IFtmLm1vbnRobHlfd2FnZSBmb3IgZiBpbiBtb2RlbC5maXJtc10KICAgICAgICBtYXJrZXRfd2FnZSA9IGZsb2F0KG5wLm1lYW4oYWxsX3dhZ2VzKSkKICAgICAgICB3YWdlX3ZzX21rdCA9IGZsb2F0KG5wLnRhbmgoKGZpcm0ubW9udGhseV93YWdlIC0gbWFya2V0X3dhZ2UpIC8gbWF4KG1hcmtldF93YWdlLCAxLjApKSkKCiAgICAgICAgbGFib3JfcmF0aW8gICA9IGxhYm9yIC8gNDAuMAogICAgICAgIHZhY2FuY3lfcmF0aW8gPSBtaW4oZmlybS52YWNhbmNpZXMsIDUpIC8gNS4wCiAgICAgICAgd29ya2VyX2NoYW5nZSA9IGZsb2F0KG5wLnRhbmgoCiAgICAgICAgICAgIChsYWJvciAtIHNlbGYucHJldl93b3JrZXJzW2Zpcm0udWlkXSkgLyAzLjAKICAgICAgICApKQogICAgICAgIHdhZ2VfY2xvY2sgPSAoc2VsZi5jdXJyZW50X3N0ZXAgJSAxMikgLyAxMS4wCgogICAgICAgIGF2Z19wcm9kICAgID0gZmxvYXQobnAubWVhbihbZi5wcm9kdWN0aXZpdHkgZm9yIGYgaW4gbW9kZWwuZmlybXNdKSkKICAgICAgICBhdmdfY2FwICAgICA9IGZsb2F0KG5wLm1lYW4oW2YuY2FwaXRhbCAgICAgIGZvciBmIGluIG1vZGVsLmZpcm1zXSkpCiAgICAgICAgcHJvZF92c19ta3QgPSBmbG9hdChucC50YW5oKChmaXJtLnByb2R1Y3Rpdml0eSAtIGF2Z19wcm9kKSAvIG1heChhdmdfcHJvZCwgMS4wKSkpCiAgICAgICAgY2FwX3ZzX21rdCAgPSBmbG9hdChucC50YW5oKChmaXJtLmNhcGl0YWwgICAgICAtIGF2Z19jYXApICAvIG1heChhdmdfY2FwLCAgMS4wKSkpCgogICAgICAgIHN1cnZpdmFsX3NpZ25hbCA9IGZsb2F0KG5wLnRhbmgoZmlybS5kZWZpY2l0X21vbnRocyAvIDEyLjApKQoKICAgICAgICBwZWVyX3Byb2ZpdHMgICAgICAgPSBbZi5wcm9maXQgZm9yIGYgaW4gc2VsZi5ybF9maXJtc10KICAgICAgICB0ZWFtX3Byb2ZpdF9zaWduYWwgPSBmbG9hdChucC50YW5oKGZsb2F0KG5wLm1lYW4ocGVlcl9wcm9maXRzKSkgLyA1XzAwMCkpCgogICAgICAgIHBhdGllbmNlID0gc2VsZi5tb2RlbC5tYXJrZXRfcXVpdF9wYXRpZW5jZQogICAgICAgIGF0X3Jpc2sgID0gc3VtKDEgZm9yIHcgaW4gZmlybS5jdXJyZW50X3dvcmtlcnMKICAgICAgICAgICAgICAgICAgICAgICBpZiB3Lm1vbnRoc19iZWxvd19ta3QgPj0gcGF0aWVuY2UgLy8gMikKICAgICAgICBhdF9yaXNrX2ZyYWN0aW9uID0gYXRfcmlzayAvIG1heChsYWJvciwgMSkKCiAgICAgICAgb2JzID0gbnAuYXJyYXkoWwogICAgICAgICAgICBwcm9maXRfc2lnbmFsLAogICAgICAgICAgICBwcm9maXRfY2hhbmdlX3NpZ25hbCwKICAgICAgICAgICAgdm1wbF9nYXAsCiAgICAgICAgICAgIHdhZ2VfdnNfbWt0LAogICAgICAgICAgICBsYWJvcl9yYXRpbywKICAgICAgICAgICAgdmFjYW5jeV9yYXRpbywKICAgICAgICAgICAgd29ya2VyX2NoYW5nZSwKICAgICAgICAgICAgd2FnZV9jbG9jaywKICAgICAgICAgICAgcHJvZF92c19ta3QsCiAgICAgICAgICAgIGNhcF92c19ta3QsCiAgICAgICAgICAgIHN1cnZpdmFsX3NpZ25hbCwKICAgICAgICAgICAgdGVhbV9wcm9maXRfc2lnbmFsLAogICAgICAgICAgICBhdF9yaXNrX2ZyYWN0aW9uLAogICAgICAgIF0sIGR0eXBlPW5wLmZsb2F0MzIpCiAgICAgICAgcmV0dXJuIG5wLmNsaXAob2JzLCAtMS41LCAxLjUpCgogICAgIyDilIDilIAgQWN0aW9uIG1hc2sg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACgogICAgZGVmIGFjdGlvbl9tYXNrcyhzZWxmKSAtPiBucC5uZGFycmF5OgogICAgICAgIHdhZ2Vfc3RlcCA9IChzZWxmLmN1cnJlbnRfc3RlcCAlIDEyID09IDApCiAgICAgICAgcmV0dXJuIG5wLmFycmF5KAogICAgICAgICAgICBbVHJ1ZSwgd2FnZV9zdGVwLCB3YWdlX3N0ZXAsIHdhZ2Vfc3RlcCwgd2FnZV9zdGVwLCBUcnVlLCBUcnVlLCBUcnVlXSwKICAgICAgICAgICAgZHR5cGU9Ym9vbCwKICAgICAgICApCgogICAgIyDilIDilIAgU3RlcCDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKCiAgICBkZWYgc3RlcChzZWxmLCBhY3Rpb24pOgogICAgICAgIGZpcm0gPSBzZWxmLnJsX2Zpcm1zW3NlbGYuY3VycmVudF9pZHhdCgogICAgICAgIGlmIHNlbGYuY3VycmVudF9pZHggPT0gMDoKICAgICAgICAgICAgZm9yIGYgaW4gc2VsZi5ybF9maXJtczoKICAgICAgICAgICAgICAgIHNlbGYucHJldl9wcm9maXRbZi51aWRdICA9IGYucHJvZml0CiAgICAgICAgICAgICAgICBzZWxmLnByZXZfd29ya2Vyc1tmLnVpZF0gPSBsZW4oZi5jdXJyZW50X3dvcmtlcnMpCgogICAgICAgIGZpcm0ucmxfYWN0aW9uID0gYWN0aW9uCgogICAgICAgIHNlbGYuY3VycmVudF9pZHggPSAoc2VsZi5jdXJyZW50X2lkeCArIDEpICUgTl9STF9GSVJNUwoKICAgICAgICBpZiBzZWxmLmN1cnJlbnRfaWR4ID09IDA6CiAgICAgICAgICAgIHNlbGYubW9kZWwuc3RlcCgpCiAgICAgICAgICAgIHNlbGYuY3VycmVudF9zdGVwICs9IDEKCiAgICAgICAgIyBDb29wZXJhdGl2ZTogc2hhcmVkIHJld2FyZCA9IG1lYW4gcHJvZml0IG9mIGFsbCBSTCBmaXJtcwogICAgICAgIHJld2FyZCA9IGZsb2F0KG5wLm1lYW4oW2YucmV3YXJkIGZvciBmIGluIHNlbGYucmxfZmlybXNdKSkKCiAgICAgICAgb2JzID0gc2VsZi5fb2JzZXJ2ZShzZWxmLmN1cnJlbnRfaWR4KQoKICAgICAgICB0ZXJtaW5hdGVkID0gRmFsc2UKICAgICAgICB0cnVuY2F0ZWQgID0gKHNlbGYuY3VycmVudF9pZHggPT0gMCkgYW5kIChzZWxmLmN1cnJlbnRfc3RlcCA+PSBzZWxmLm1heF9zdGVwKQogICAgICAgIGlmIHRydW5jYXRlZDoKICAgICAgICAgICAgc2VsZi5jdXJyZW50X3N0ZXAgPSAwCgogICAgICAgIHJldHVybiBvYnMsIHJld2FyZCwgdGVybWluYXRlZCwgdHJ1bmNhdGVkLCB7fQoKICAgICMg4pSA4pSAIFJlc2V0IOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAoKICAgIGRlZiByZXNldChzZWxmLCBzZWVkPU5vbmUsIG9wdGlvbnM9Tm9uZSk6CiAgICAgICAgaW1wb3J0IHJhbmRvbSBhcyBfcmFuZG9tCiAgICAgICAgZW52X3NlZWQgPSBzZWVkIGlmIHNlZWQgaXMgbm90IE5vbmUgZWxzZSBpbnQobnAucmFuZG9tLnJhbmRpbnQoMCwgMioqMzEpKQogICAgICAgIF9yYW5kb20uc2VlZChlbnZfc2VlZCkKICAgICAgICBucC5yYW5kb20uc2VlZChlbnZfc2VlZCkKCiAgICAgICAgc2VsZi5tb2RlbCAgICAgICAgPSBMYWJvck1hcmtldE1vZGVsKG5fcmxfZmlybXM9Tl9STF9GSVJNUywKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgdXNlX3dhZ2VfZ2FwX3Byb2I9VHJ1ZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgZXF1YWxfdGVybXM9VHJ1ZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgc2VlZD1lbnZfc2VlZCkKICAgICAgICBzZWxmLnJsX2Zpcm1zICAgICA9IHNlbGYubW9kZWwuZmlybXNbOk5fUkxfRklSTVNdCiAgICAgICAgc2VsZi5jdXJyZW50X2lkeCAgPSAwCiAgICAgICAgc2VsZi5jdXJyZW50X3N0ZXAgPSAwCiAgICAgICAgc2VsZi5wcmV2X3Byb2ZpdCAgPSB7Zi51aWQ6IDAuMCAgICAgICAgICAgICAgICAgICAgZm9yIGYgaW4gc2VsZi5ybF9maXJtc30KICAgICAgICBzZWxmLnByZXZfd29ya2VycyA9IHtmLnVpZDogbGVuKGYuY3VycmVudF93b3JrZXJzKSBmb3IgZiBpbiBzZWxmLnJsX2Zpcm1zfQogICAgICAgIHJldHVybiBzZWxmLl9vYnNlcnZlKDApLCB7fQo=", "comp_env": "IyBjb21wZXRpdGl2ZS9maXJtX2Vudi5weQojCiMgTXVsdGktYWdlbnQgZW52OiBOX1JMX0ZJUk1TIGZpcm1zIGNvbXBldGUgYWdhaW5zdCBlYWNoIG90aGVyLgojIENPTVBFVElUSVZFIG1vZGU6IHJld2FyZCA9IG93bl9yZXdhcmQgLSAwLjUgKiBtZWFuKG90aGVyIFJMIHJld2FyZHMpLgojIEVhY2ggZmlybSBpcyBpbmNlbnRpdmlzZWQgdG8gYmVhdCBpdHMgUkwgcGVlcnMsIG5vdCBqdXN0IGVhcm4gcHJvZml0LgojCiMgT2JzZXJ2YXRpb24gKDEzIGZlYXR1cmVzKToKIyAgIDAgIHByb2ZpdF9zaWduYWwgICAgICAg4oCUIHRhbmgocHJvZml0LzVrKSAgICAgICAgICAgICAgb3duIHByb2ZpdCBsZXZlbAojICAgMSAgcHJvZml0X2NoYW5nZSAgICAgICDigJQgdGFuaCjOlHByb2ZpdC8yaykgICAgICAgICAgICAgb3duIHByb2ZpdCB0cmVuZAojICAgMiAgdm1wbF9nYXAgICAgICAgICAgICDigJQgdGFuaCgoVk1QTC13YWdlKS93YWdlKSAgICAgICAgb3Zlci91bmRlcnBheWluZyB2cyBNUEwKIyAgIDMgIHdhZ2VfdnNfbWt0ICAgICAgICAg4oCUIHRhbmgoKG93bi1ta3QpL21rdCkgICAgICAgICAgd2FnZSB2cyBBTEwgZmlybXMnIGF2ZwojICAgNCAgbGFib3JfcmF0aW8gICAgICAgICDigJQgbl93b3JrZXJzLzQwICAgICAgICAgICAgICAgICB3b3JrZm9yY2Ugc2l6ZQojICAgNSAgdmFjYW5jeV9yYXRpbyAgICAgICDigJQgbWluKHZhYyw1KS81ICAgICAgICAgICAgICAgICBvcGVuIHNsb3RzCiMgICA2ICB3b3JrZXJfY2hhbmdlICAgICAgIOKAlCB0YW5oKM6Ud29ya2Vycy8zKSAgICAgICAgICAgICByZWNlbnQgaGVhZGNvdW50IM6UCiMgICA3ICB3YWdlX2Nsb2NrICAgICAgICAgIOKAlCBzdGVwJTEyLzExICAgICAgICAgICAgICAgICAgIHBvc2l0aW9uIGluIHdhZ2UgY3ljbGUKIyAgIDggIHByb2RfdnNfbWt0ICAgICAgICAg4oCUIHRhbmgoKEEtYXZnQSkvYXZnQSkgICAgICAgICAgb3duIHByb2R1Y3Rpdml0eSB2cyBBTEwKIyAgIDkgIGNhcF92c19ta3QgICAgICAgICAg4oCUIHRhbmgoKEstYXZnSykvYXZnSykgICAgICAgICAgb3duIGNhcGl0YWwgdnMgQUxMCiMgIDEwICBwcm9maXRfdnNfcGVlcnMgICAgIOKAlCB0YW5oKChvd24tUkxfYXZnX3Byb2ZpdCkvNWspIGFtIEkgYmVhdGluZyBwZWVycz8KIyAgMTEgIHdhZ2VfdnNfcGVlcnMgICAgICAg4oCUIHRhbmgoKG93bi1STF9hdmdfd2FnZSkvUkxfYXZnKSBhbSBJIG91dGJpZGRpbmcgcGVlcnM/CiMgIDEyICBhdF9yaXNrX2ZyYWN0aW9uICAgIOKAlCB3b3JrZXJzIHcvIG1vbnRoc19iZWxvd19ta3Q+PXBhdGllbmNlLy8yIC8gbWF4KGxhYm9yLDEpCgppbXBvcnQgZ3ltbmFzaXVtIGFzIGd5bQppbXBvcnQgbnVtcHkgYXMgbnAKZnJvbSBtb2RlbF9ybCBpbXBvcnQgTGFib3JNYXJrZXRNb2RlbAoKTl9STF9GSVJNUyA9IDMKCgpjbGFzcyBDb21wRmlybUVudihneW0uRW52KToKCiAgICBkZWYgX19pbml0X18oc2VsZik6CiAgICAgICAgc2VsZi5tb2RlbCAgICAgICAgPSBMYWJvck1hcmtldE1vZGVsKG5fcmxfZmlybXM9Tl9STF9GSVJNUywgZXF1YWxfdGVybXM9VHJ1ZSkKICAgICAgICBzZWxmLnJsX2Zpcm1zICAgICA9IHNlbGYubW9kZWwuZmlybXNbOk5fUkxfRklSTVNdCiAgICAgICAgc2VsZi5jdXJyZW50X2lkeCAgPSAwCiAgICAgICAgc2VsZi5jdXJyZW50X3N0ZXAgPSAwCiAgICAgICAgc2VsZi5tYXhfc3RlcCAgICAgPSAzNjAKCiAgICAgICAgc2VsZi5wcmV2X3Byb2ZpdCAgPSB7Zi51aWQ6IDAuMCAgICAgICAgICAgICAgICAgICAgZm9yIGYgaW4gc2VsZi5ybF9maXJtc30KICAgICAgICBzZWxmLnByZXZfd29ya2VycyA9IHtmLnVpZDogbGVuKGYuY3VycmVudF93b3JrZXJzKSBmb3IgZiBpbiBzZWxmLnJsX2Zpcm1zfQoKICAgICAgICBzZWxmLmFjdGlvbl9zcGFjZSA9IGd5bS5zcGFjZXMuRGlzY3JldGUoOCkKICAgICAgICBzZWxmLm9ic2VydmF0aW9uX3NwYWNlID0gZ3ltLnNwYWNlcy5Cb3goCiAgICAgICAgICAgIGxvdz0tMS41LCBoaWdoPTEuNSwgc2hhcGU9KDEzLCksIGR0eXBlPW5wLmZsb2F0MzIKICAgICAgICApCgogICAgIyDilIDilIAgT2JzZXJ2YXRpb24gZm9yIG9uZSBSTCBmaXJtIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAoKICAgIGRlZiBfb2JzZXJ2ZShzZWxmLCBpZHgpOgogICAgICAgIGZpcm0gID0gc2VsZi5ybF9maXJtc1tpZHhdCiAgICAgICAgbW9kZWwgPSBzZWxmLm1vZGVsCgogICAgICAgIHByb2ZpdF9zaWduYWwgICAgICAgID0gZmxvYXQobnAudGFuaChmaXJtLnByb2ZpdCAvIDVfMDAwKSkKICAgICAgICBwcm9maXRfY2hhbmdlX3NpZ25hbCA9IGZsb2F0KG5wLnRhbmgoCiAgICAgICAgICAgIChmaXJtLnByb2ZpdCAtIHNlbGYucHJldl9wcm9maXRbZmlybS51aWRdKSAvIDJfMDAwCiAgICAgICAgKSkKCiAgICAgICAgbGFib3IgPSBsZW4oZmlybS5jdXJyZW50X3dvcmtlcnMpCiAgICAgICAgaWYgbGFib3IgPiAwOgogICAgICAgICAgICBtcGwgICAgICA9IGZpcm0ubWFyZ2luYWxfcHJvZHVjdF9sYWJvcihmaXJtLnByb2R1Y3Rpdml0eSwgbGFib3IsIGZpcm0uYWxwaGEpCiAgICAgICAgICAgIHZtcGwgICAgID0gbXBsICogZmlybS5vdXRwdXRfcHJpY2UKICAgICAgICAgICAgdm1wbF9nYXAgPSBmbG9hdChucC50YW5oKCh2bXBsIC0gZmlybS5tb250aGx5X3dhZ2UpIC8gbWF4KGZpcm0ubW9udGhseV93YWdlLCAxLjApKSkKICAgICAgICBlbHNlOgogICAgICAgICAgICB2bXBsX2dhcCA9IDEuMAoKICAgICAgICBhbGxfd2FnZXMgICA9IFtmLm1vbnRobHlfd2FnZSBmb3IgZiBpbiBtb2RlbC5maXJtc10KICAgICAgICBtYXJrZXRfd2FnZSA9IGZsb2F0KG5wLm1lYW4oYWxsX3dhZ2VzKSkKICAgICAgICB3YWdlX3ZzX21rdCA9IGZsb2F0KG5wLnRhbmgoKGZpcm0ubW9udGhseV93YWdlIC0gbWFya2V0X3dhZ2UpIC8gbWF4KG1hcmtldF93YWdlLCAxLjApKSkKCiAgICAgICAgbGFib3JfcmF0aW8gICA9IGxhYm9yIC8gNDAuMAogICAgICAgIHZhY2FuY3lfcmF0aW8gPSBtaW4oZmlybS52YWNhbmNpZXMsIDUpIC8gNS4wCiAgICAgICAgd29ya2VyX2NoYW5nZSA9IGZsb2F0KG5wLnRhbmgoCiAgICAgICAgICAgIChsYWJvciAtIHNlbGYucHJldl93b3JrZXJzW2Zpcm0udWlkXSkgLyAzLjAKICAgICAgICApKQogICAgICAgIHdhZ2VfY2xvY2sgPSAoc2VsZi5jdXJyZW50X3N0ZXAgJSAxMikgLyAxMS4wCgogICAgICAgIGF2Z19wcm9kICAgID0gZmxvYXQobnAubWVhbihbZi5wcm9kdWN0aXZpdHkgZm9yIGYgaW4gbW9kZWwuZmlybXNdKSkKICAgICAgICBhdmdfY2FwICAgICA9IGZsb2F0KG5wLm1lYW4oW2YuY2FwaXRhbCAgICAgIGZvciBmIGluIG1vZGVsLmZpcm1zXSkpCiAgICAgICAgcHJvZF92c19ta3QgPSBmbG9hdChucC50YW5oKChmaXJtLnByb2R1Y3Rpdml0eSAtIGF2Z19wcm9kKSAvIG1heChhdmdfcHJvZCwgMS4wKSkpCiAgICAgICAgY2FwX3ZzX21rdCAgPSBmbG9hdChucC50YW5oKChmaXJtLmNhcGl0YWwgICAgICAtIGF2Z19jYXApICAvIG1heChhdmdfY2FwLCAgMS4wKSkpCgogICAgICAgICMgUGVlciBjb21wYXJpc29uIHNpZ25hbHMgKGNvbXBldGl0aXZlIGFkdmFudGFnZSkKICAgICAgICBwZWVycyA9IFtmIGZvciBmIGluIHNlbGYucmxfZmlybXMgaWYgZiBpcyBub3QgZmlybV0KCiAgICAgICAgcGVlcl9hdmdfcHJvZml0ID0gZmxvYXQobnAubWVhbihbZi5wcm9maXQgICAgICAgZm9yIGYgaW4gcGVlcnNdKSkgaWYgcGVlcnMgZWxzZSAwLjAKICAgICAgICBwZWVyX2F2Z193YWdlICAgPSBmbG9hdChucC5tZWFuKFtmLm1vbnRobHlfd2FnZSAgZm9yIGYgaW4gcGVlcnNdKSkgaWYgcGVlcnMgZWxzZSBmaXJtLm1vbnRobHlfd2FnZQoKICAgICAgICBwcm9maXRfdnNfcGVlcnMgPSBmbG9hdChucC50YW5oKChmaXJtLnByb2ZpdCAtIHBlZXJfYXZnX3Byb2ZpdCkgLyA1XzAwMCkpCiAgICAgICAgd2FnZV92c19wZWVycyAgID0gZmxvYXQobnAudGFuaCgKICAgICAgICAgICAgKGZpcm0ubW9udGhseV93YWdlIC0gcGVlcl9hdmdfd2FnZSkgLyBtYXgocGVlcl9hdmdfd2FnZSwgMS4wKQogICAgICAgICkpCgogICAgICAgIHBhdGllbmNlID0gc2VsZi5tb2RlbC5tYXJrZXRfcXVpdF9wYXRpZW5jZQogICAgICAgIGF0X3Jpc2sgID0gc3VtKDEgZm9yIHcgaW4gZmlybS5jdXJyZW50X3dvcmtlcnMKICAgICAgICAgICAgICAgICAgICAgICBpZiB3Lm1vbnRoc19iZWxvd19ta3QgPj0gcGF0aWVuY2UgLy8gMikKICAgICAgICBhdF9yaXNrX2ZyYWN0aW9uID0gYXRfcmlzayAvIG1heChsYWJvciwgMSkKCiAgICAgICAgb2JzID0gbnAuYXJyYXkoWwogICAgICAgICAgICBwcm9maXRfc2lnbmFsLAogICAgICAgICAgICBwcm9maXRfY2hhbmdlX3NpZ25hbCwKICAgICAgICAgICAgdm1wbF9nYXAsCiAgICAgICAgICAgIHdhZ2VfdnNfbWt0LAogICAgICAgICAgICBsYWJvcl9yYXRpbywKICAgICAgICAgICAgdmFjYW5jeV9yYXRpbywKICAgICAgICAgICAgd29ya2VyX2NoYW5nZSwKICAgICAgICAgICAgd2FnZV9jbG9jaywKICAgICAgICAgICAgcHJvZF92c19ta3QsCiAgICAgICAgICAgIGNhcF92c19ta3QsCiAgICAgICAgICAgIHByb2ZpdF92c19wZWVycywKICAgICAgICAgICAgd2FnZV92c19wZWVycywKICAgICAgICAgICAgYXRfcmlza19mcmFjdGlvbiwKICAgICAgICBdLCBkdHlwZT1ucC5mbG9hdDMyKQogICAgICAgIHJldHVybiBucC5jbGlwKG9icywgLTEuNSwgMS41KQoKICAgICMg4pSA4pSAIEFjdGlvbiBtYXNrIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAoKICAgIGRlZiBhY3Rpb25fbWFza3Moc2VsZikgLT4gbnAubmRhcnJheToKICAgICAgICB3YWdlX3N0ZXAgPSAoc2VsZi5jdXJyZW50X3N0ZXAgJSAxMiA9PSAwKQogICAgICAgIHJldHVybiBucC5hcnJheSgKICAgICAgICAgICAgW1RydWUsIHdhZ2Vfc3RlcCwgd2FnZV9zdGVwLCB3YWdlX3N0ZXAsIHdhZ2Vfc3RlcCwgVHJ1ZSwgVHJ1ZSwgVHJ1ZV0sCiAgICAgICAgICAgIGR0eXBlPWJvb2wsCiAgICAgICAgKQoKICAgICMg4pSA4pSAIFN0ZXAg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACgogICAgZGVmIHN0ZXAoc2VsZiwgYWN0aW9uKToKICAgICAgICBmaXJtID0gc2VsZi5ybF9maXJtc1tzZWxmLmN1cnJlbnRfaWR4XQoKICAgICAgICBpZiBzZWxmLmN1cnJlbnRfaWR4ID09IDA6CiAgICAgICAgICAgIGZvciBmIGluIHNlbGYucmxfZmlybXM6CiAgICAgICAgICAgICAgICBzZWxmLnByZXZfcHJvZml0W2YudWlkXSAgPSBmLnByb2ZpdAogICAgICAgICAgICAgICAgc2VsZi5wcmV2X3dvcmtlcnNbZi51aWRdID0gbGVuKGYuY3VycmVudF93b3JrZXJzKQoKICAgICAgICBmaXJtLnJsX2FjdGlvbiA9IGFjdGlvbgoKICAgICAgICBzZWxmLmN1cnJlbnRfaWR4ID0gKHNlbGYuY3VycmVudF9pZHggKyAxKSAlIE5fUkxfRklSTVMKCiAgICAgICAgaWYgc2VsZi5jdXJyZW50X2lkeCA9PSAwOgogICAgICAgICAgICBzZWxmLm1vZGVsLnN0ZXAoKQogICAgICAgICAgICBzZWxmLmN1cnJlbnRfc3RlcCArPSAxCgogICAgICAgICMgQ29tcGV0aXRpdmU6IG93biByZXdhcmQgbWludXMgaGFsZiBvZiBwZWVycycgYXZlcmFnZSByZXdhcmQKICAgICAgICBwZWVyc19yZXdhcmQgPSBmbG9hdChucC5tZWFuKFsKICAgICAgICAgICAgZi5yZXdhcmQgZm9yIGYgaW4gc2VsZi5ybF9maXJtcyBpZiBmIGlzIG5vdCBmaXJtCiAgICAgICAgXSkpIGlmIE5fUkxfRklSTVMgPiAxIGVsc2UgMC4wCiAgICAgICAgcmV3YXJkID0gZmlybS5yZXdhcmQgLSAwLjUgKiBwZWVyc19yZXdhcmQKCiAgICAgICAgb2JzID0gc2VsZi5fb2JzZXJ2ZShzZWxmLmN1cnJlbnRfaWR4KQoKICAgICAgICB0ZXJtaW5hdGVkID0gRmFsc2UKICAgICAgICB0cnVuY2F0ZWQgID0gKHNlbGYuY3VycmVudF9pZHggPT0gMCkgYW5kIChzZWxmLmN1cnJlbnRfc3RlcCA+PSBzZWxmLm1heF9zdGVwKQogICAgICAgIGlmIHRydW5jYXRlZDoKICAgICAgICAgICAgc2VsZi5jdXJyZW50X3N0ZXAgPSAwCgogICAgICAgIHJldHVybiBvYnMsIHJld2FyZCwgdGVybWluYXRlZCwgdHJ1bmNhdGVkLCB7fQoKICAgICMg4pSA4pSAIFJlc2V0IOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAoKICAgIGRlZiByZXNldChzZWxmLCBzZWVkPU5vbmUsIG9wdGlvbnM9Tm9uZSk6CiAgICAgICAgaW1wb3J0IHJhbmRvbSBhcyBfcmFuZG9tCiAgICAgICAgZW52X3NlZWQgPSBzZWVkIGlmIHNlZWQgaXMgbm90IE5vbmUgZWxzZSBpbnQobnAucmFuZG9tLnJhbmRpbnQoMCwgMioqMzEpKQogICAgICAgIF9yYW5kb20uc2VlZChlbnZfc2VlZCkKICAgICAgICBucC5yYW5kb20uc2VlZChlbnZfc2VlZCkKCiAgICAgICAgc2VsZi5tb2RlbCAgICAgICAgPSBMYWJvck1hcmtldE1vZGVsKG5fcmxfZmlybXM9Tl9STF9GSVJNUywKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgdXNlX3dhZ2VfZ2FwX3Byb2I9VHJ1ZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgZXF1YWxfdGVybXM9VHJ1ZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgc2VlZD1lbnZfc2VlZCkKICAgICAgICBzZWxmLnJsX2Zpcm1zICAgICA9IHNlbGYubW9kZWwuZmlybXNbOk5fUkxfRklSTVNdCiAgICAgICAgc2VsZi5jdXJyZW50X2lkeCAgPSAwCiAgICAgICAgc2VsZi5jdXJyZW50X3N0ZXAgPSAwCiAgICAgICAgc2VsZi5wcmV2X3Byb2ZpdCAgPSB7Zi51aWQ6IDAuMCAgICAgICAgICAgICAgICAgICAgZm9yIGYgaW4gc2VsZi5ybF9maXJtc30KICAgICAgICBzZWxmLnByZXZfd29ya2VycyA9IHtmLnVpZDogbGVuKGYuY3VycmVudF93b3JrZXJzKSBmb3IgZiBpbiBzZWxmLnJsX2Zpcm1zfQogICAgICAgIHJldHVybiBzZWxmLl9vYnNlcnZlKDApLCB7fQo=", "coop_vis": "IyBjb29wZXJhdGl2ZS9ybF92aXMucHkg4oCUIHRyYWluaW5nIGNhbGxiYWNrIGZvciBjb29wZXJhdGl2ZSBzY2VuYXJpbw0KDQppbXBvcnQgc2h1dGlsDQppbXBvcnQgbnVtcHkgYXMgbnANCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aA0KZnJvbSBzdGFibGVfYmFzZWxpbmVzMy5jb21tb24uY2FsbGJhY2tzIGltcG9ydCBCYXNlQ2FsbGJhY2sNCg0KQUNUSU9OX05BTUVTID0gew0KICAgIDA6ICJob2xkIiwgMTogIndhZ2VfdXBfMzAwIiwgMjogIndhZ2VfdXBfMTAwIiwNCiAgICAzOiAid2FnZV9kbl8xMDAiLCA0OiAid2FnZV9kbl8zMDAiLCA1OiAicG9zdF92YWNhbmN5IiwNCiAgICA2OiAiZmlyZV93b3JrZXIiLCA3OiAic25hcF90b19tYXJrZXQiLA0KfQ0KDQoNCmNsYXNzIExhYm9yTWV0cmljc0NhbGxiYWNrKEJhc2VDYWxsYmFjayk6DQoNCiAgICBkZWYgX19pbml0X18oc2VsZiwgbG9nX2Rpcj0iLi90ZW5zb3Jib2FyZF9sb2dzIiwgYWxnb19uYW1lPSJDb29wX01hc2thYmxlUFBPIiwNCiAgICAgICAgICAgICAgICAga2VlcF9ydW5zPTMsIHZlcmJvc2U9MCk6DQogICAgICAgIHN1cGVyKCkuX19pbml0X18odmVyYm9zZSkNCiAgICAgICAgc2VsZi5sb2dfZGlyICAgICAgID0gUGF0aChsb2dfZGlyKQ0KICAgICAgICBzZWxmLmFsZ29fbmFtZSAgICAgPSBhbGdvX25hbWUNCiAgICAgICAgc2VsZi5rZWVwX3J1bnMgICAgID0ga2VlcF9ydW5zDQogICAgICAgIHNlbGYuYWN0aW9uX2NvdW50cyA9IG5wLnplcm9zKGxlbihBQ1RJT05fTkFNRVMpLCBkdHlwZT1ucC5pbnQ2NCkNCg0KICAgIGRlZiBfb25fdHJhaW5pbmdfc3RhcnQoc2VsZik6DQogICAgICAgIHRyeToNCiAgICAgICAgICAgIHJ1bnMgPSBbXQ0KICAgICAgICAgICAgZm9yIGQgaW4gc2VsZi5sb2dfZGlyLml0ZXJkaXIoKToNCiAgICAgICAgICAgICAgICBpZiBkLmlzX2RpcigpIGFuZCBkLm5hbWUuc3RhcnRzd2l0aChzZWxmLmFsZ29fbmFtZSArICJfIik6DQogICAgICAgICAgICAgICAgICAgIHRyeToNCiAgICAgICAgICAgICAgICAgICAgICAgIG51bSA9IGludChkLm5hbWUucnNwbGl0KCJfIiwgMSlbLTFdKQ0KICAgICAgICAgICAgICAgICAgICAgICAgcnVucy5hcHBlbmQoKG51bSwgZCkpDQogICAgICAgICAgICAgICAgICAgIGV4Y2VwdCBWYWx1ZUVycm9yOg0KICAgICAgICAgICAgICAgICAgICAgICAgcGFzcw0KICAgICAgICAgICAgcnVucy5zb3J0KGtleT1sYW1iZGEgeDogeFswXSkNCiAgICAgICAgICAgIHRvX2RlbGV0ZSA9IHJ1bnNbOiAtc2VsZi5rZWVwX3J1bnNdIGlmIGxlbihydW5zKSA+IHNlbGYua2VlcF9ydW5zIGVsc2UgW10NCiAgICAgICAgICAgIGZvciBfLCBydW5fZGlyIGluIHRvX2RlbGV0ZToNCiAgICAgICAgICAgICAgICBzaHV0aWwucm10cmVlKHJ1bl9kaXIpDQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb246DQogICAgICAgICAgICBwYXNzDQoNCiAgICBkZWYgX29uX3N0ZXAoc2VsZikgLT4gYm9vbDoNCiAgICAgICAgZW52ID0gc2VsZi50cmFpbmluZ19lbnYuZW52c1swXS5lbnYgICAjIHVud3JhcCBBY3Rpb25NYXNrZXIg4oaSIENvb3BGaXJtRW52DQogICAgICAgIHNpbSA9IGVudi5tb2RlbA0KDQogICAgICAgIGFjdGlvbnMgPSBzZWxmLmxvY2Fscy5nZXQoImFjdGlvbnMiKQ0KICAgICAgICBpZiBhY3Rpb25zIGlzIG5vdCBOb25lOg0KICAgICAgICAgICAgc2VsZi5hY3Rpb25fY291bnRzW2ludChhY3Rpb25zWzBdKV0gKz0gMQ0KDQogICAgICAgICMgVHJhY2sgYWxsIFJMIGZpcm1zDQogICAgICAgIHJsX3Byb2ZpdHMgID0gW2YucHJvZml0ICBmb3IgZiBpbiBlbnYucmxfZmlybXNdDQogICAgICAgIHJsX3dhZ2VzICAgID0gW2YubW9udGhseV93YWdlIGZvciBmIGluIGVudi5ybF9maXJtc10NCiAgICAgICAgcmxfd29ya2VycyAgPSBbbGVuKGYuY3VycmVudF93b3JrZXJzKSBmb3IgZiBpbiBlbnYucmxfZmlybXNdDQoNCiAgICAgICAgc2VsZi5sb2dnZXIucmVjb3JkKCJjb29wL2F2Z19ybF9wcm9maXQiLCAgICBmbG9hdChucC5tZWFuKHJsX3Byb2ZpdHMpKSkNCiAgICAgICAgc2VsZi5sb2dnZXIucmVjb3JkKCJjb29wL2F2Z19ybF93YWdlIiwgICAgICBmbG9hdChucC5tZWFuKHJsX3dhZ2VzKSkpDQogICAgICAgIHNlbGYubG9nZ2VyLnJlY29yZCgiY29vcC9hdmdfcmxfd29ya2VycyIsICAgZmxvYXQobnAubWVhbihybF93b3JrZXJzKSkpDQogICAgICAgIHNlbGYubG9nZ2VyLnJlY29yZCgiY29vcC93YWdlX3NwcmVhZCIsICAgICAgZmxvYXQobnAubWF4KHJsX3dhZ2VzKSAtIG5wLm1pbihybF93YWdlcykpKQ0KICAgICAgICBzZWxmLmxvZ2dlci5yZWNvcmQoImNvb3AvYXZnX2RlZmljaXRfbW9udGhzIiwNCiAgICAgICAgICAgICAgICAgICAgICAgICAgIGZsb2F0KG5wLm1lYW4oW2YuZGVmaWNpdF9tb250aHMgZm9yIGYgaW4gZW52LnJsX2Zpcm1zXSkpKQ0KDQogICAgICAgICMgRWNvbm9teQ0KICAgICAgICBlbXBsb3llZCAgICA9IFt3IGZvciB3IGluIHNpbS53b3JrZXJzIGlmIHcuZW1wbG95ZWRdDQogICAgICAgIGVtcGxveV9yYXRlID0gbGVuKGVtcGxveWVkKSAvIGxlbihzaW0ud29ya2VycykgaWYgc2ltLndvcmtlcnMgZWxzZSAwDQogICAgICAgIGF2Z19wcm9maXQgID0gZmxvYXQobnAubWVhbihbZi5wcm9maXQgZm9yIGYgaW4gc2ltLmZpcm1zXSkpDQogICAgICAgIG1hcmtldF93YWdlID0gZmxvYXQobnAubWVhbihbZi5tb250aGx5X3dhZ2UgZm9yIGYgaW4gc2ltLmZpcm1zXSkpDQoNCiAgICAgICAgc2VsZi5sb2dnZXIucmVjb3JkKCJlY29ub215L2VtcGxveW1lbnRfcmF0ZSIsICBlbXBsb3lfcmF0ZSkNCiAgICAgICAgc2VsZi5sb2dnZXIucmVjb3JkKCJlY29ub215L2F2Z19wcm9maXRfYWxsIiwgICBhdmdfcHJvZml0KQ0KICAgICAgICBzZWxmLmxvZ2dlci5yZWNvcmQoImVjb25vbXkvbWFya2V0X2F2Z193YWdlIiwgIG1hcmtldF93YWdlKQ0KDQogICAgICAgICMgUkwgdnMgaGV1cmlzdGljDQogICAgICAgIGhldXJpc3RpYyA9IFtmIGZvciBmIGluIHNpbS5maXJtcyBpZiBmIG5vdCBpbiBlbnYucmxfZmlybXNdDQogICAgICAgIGlmIGhldXJpc3RpYzoNCiAgICAgICAgICAgIGhfYXZnID0gZmxvYXQobnAubWVhbihbZi5wcm9maXQgZm9yIGYgaW4gaGV1cmlzdGljXSkpDQogICAgICAgICAgICBzZWxmLmxvZ2dlci5yZWNvcmQoImNvb3AvcmxfdnNfaGV1cmlzdGljIiwgZmxvYXQobnAubWVhbihybF9wcm9maXRzKSkgLSBoX2F2ZykNCg0KICAgICAgICB0b3RhbCA9IHNlbGYuYWN0aW9uX2NvdW50cy5zdW0oKQ0KICAgICAgICBpZiB0b3RhbCA+IDA6DQogICAgICAgICAgICBmb3IgaWR4LCBuYW1lIGluIEFDVElPTl9OQU1FUy5pdGVtcygpOg0KICAgICAgICAgICAgICAgIHNlbGYubG9nZ2VyLnJlY29yZChmImFjdGlvbnMve25hbWV9Iiwgc2VsZi5hY3Rpb25fY291bnRzW2lkeF0gLyB0b3RhbCkNCg0KICAgICAgICByZXR1cm4gVHJ1ZQ0K", "comp_vis": "IyBjb21wZXRpdGl2ZS9ybF92aXMucHkg4oCUIHRyYWluaW5nIGNhbGxiYWNrIGZvciBjb21wZXRpdGl2ZSBzY2VuYXJpbw0KDQppbXBvcnQgc2h1dGlsDQppbXBvcnQgbnVtcHkgYXMgbnANCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aA0KZnJvbSBzdGFibGVfYmFzZWxpbmVzMy5jb21tb24uY2FsbGJhY2tzIGltcG9ydCBCYXNlQ2FsbGJhY2sNCg0KQUNUSU9OX05BTUVTID0gew0KICAgIDA6ICJob2xkIiwgMTogIndhZ2VfdXBfMzAwIiwgMjogIndhZ2VfdXBfMTAwIiwNCiAgICAzOiAid2FnZV9kbl8xMDAiLCA0OiAid2FnZV9kbl8zMDAiLCA1OiAicG9zdF92YWNhbmN5IiwNCiAgICA2OiAiZmlyZV93b3JrZXIiLCA3OiAic25hcF90b19tYXJrZXQiLA0KfQ0KDQoNCmNsYXNzIExhYm9yTWV0cmljc0NhbGxiYWNrKEJhc2VDYWxsYmFjayk6DQoNCiAgICBkZWYgX19pbml0X18oc2VsZiwgbG9nX2Rpcj0iLi90ZW5zb3Jib2FyZF9sb2dzIiwgYWxnb19uYW1lPSJDb21wX01hc2thYmxlUFBPIiwNCiAgICAgICAgICAgICAgICAga2VlcF9ydW5zPTMsIHZlcmJvc2U9MCk6DQogICAgICAgIHN1cGVyKCkuX19pbml0X18odmVyYm9zZSkNCiAgICAgICAgc2VsZi5sb2dfZGlyICAgICAgID0gUGF0aChsb2dfZGlyKQ0KICAgICAgICBzZWxmLmFsZ29fbmFtZSAgICAgPSBhbGdvX25hbWUNCiAgICAgICAgc2VsZi5rZWVwX3J1bnMgICAgID0ga2VlcF9ydW5zDQogICAgICAgIHNlbGYuYWN0aW9uX2NvdW50cyA9IG5wLnplcm9zKGxlbihBQ1RJT05fTkFNRVMpLCBkdHlwZT1ucC5pbnQ2NCkNCg0KICAgIGRlZiBfb25fdHJhaW5pbmdfc3RhcnQoc2VsZik6DQogICAgICAgIHRyeToNCiAgICAgICAgICAgIHJ1bnMgPSBbXQ0KICAgICAgICAgICAgZm9yIGQgaW4gc2VsZi5sb2dfZGlyLml0ZXJkaXIoKToNCiAgICAgICAgICAgICAgICBpZiBkLmlzX2RpcigpIGFuZCBkLm5hbWUuc3RhcnRzd2l0aChzZWxmLmFsZ29fbmFtZSArICJfIik6DQogICAgICAgICAgICAgICAgICAgIHRyeToNCiAgICAgICAgICAgICAgICAgICAgICAgIG51bSA9IGludChkLm5hbWUucnNwbGl0KCJfIiwgMSlbLTFdKQ0KICAgICAgICAgICAgICAgICAgICAgICAgcnVucy5hcHBlbmQoKG51bSwgZCkpDQogICAgICAgICAgICAgICAgICAgIGV4Y2VwdCBWYWx1ZUVycm9yOg0KICAgICAgICAgICAgICAgICAgICAgICAgcGFzcw0KICAgICAgICAgICAgcnVucy5zb3J0KGtleT1sYW1iZGEgeDogeFswXSkNCiAgICAgICAgICAgIHRvX2RlbGV0ZSA9IHJ1bnNbOiAtc2VsZi5rZWVwX3J1bnNdIGlmIGxlbihydW5zKSA+IHNlbGYua2VlcF9ydW5zIGVsc2UgW10NCiAgICAgICAgICAgIGZvciBfLCBydW5fZGlyIGluIHRvX2RlbGV0ZToNCiAgICAgICAgICAgICAgICBzaHV0aWwucm10cmVlKHJ1bl9kaXIpDQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb246DQogICAgICAgICAgICBwYXNzDQoNCiAgICBkZWYgX29uX3N0ZXAoc2VsZikgLT4gYm9vbDoNCiAgICAgICAgZW52ID0gc2VsZi50cmFpbmluZ19lbnYuZW52c1swXS5lbnYNCiAgICAgICAgc2ltID0gZW52Lm1vZGVsDQoNCiAgICAgICAgYWN0aW9ucyA9IHNlbGYubG9jYWxzLmdldCgiYWN0aW9ucyIpDQogICAgICAgIGlmIGFjdGlvbnMgaXMgbm90IE5vbmU6DQogICAgICAgICAgICBzZWxmLmFjdGlvbl9jb3VudHNbaW50KGFjdGlvbnNbMF0pXSArPSAxDQoNCiAgICAgICAgcmxfcHJvZml0cyAgPSBbZi5wcm9maXQgICAgICAgZm9yIGYgaW4gZW52LnJsX2Zpcm1zXQ0KICAgICAgICBybF93YWdlcyAgICA9IFtmLm1vbnRobHlfd2FnZSBmb3IgZiBpbiBlbnYucmxfZmlybXNdDQogICAgICAgIHJsX3dvcmtlcnMgID0gW2xlbihmLmN1cnJlbnRfd29ya2VycykgZm9yIGYgaW4gZW52LnJsX2Zpcm1zXQ0KDQogICAgICAgIHNlbGYubG9nZ2VyLnJlY29yZCgiY29tcC9hdmdfcmxfcHJvZml0IiwgICAgZmxvYXQobnAubWVhbihybF9wcm9maXRzKSkpDQogICAgICAgIHNlbGYubG9nZ2VyLnJlY29yZCgiY29tcC9tYXhfcmxfcHJvZml0IiwgICAgZmxvYXQobnAubWF4KHJsX3Byb2ZpdHMpKSkNCiAgICAgICAgc2VsZi5sb2dnZXIucmVjb3JkKCJjb21wL21pbl9ybF9wcm9maXQiLCAgICBmbG9hdChucC5taW4ocmxfcHJvZml0cykpKQ0KICAgICAgICBzZWxmLmxvZ2dlci5yZWNvcmQoImNvbXAvcHJvZml0X3NwcmVhZCIsICAgIGZsb2F0KG5wLm1heChybF9wcm9maXRzKSAtIG5wLm1pbihybF9wcm9maXRzKSkpDQogICAgICAgIHNlbGYubG9nZ2VyLnJlY29yZCgiY29tcC9hdmdfcmxfd2FnZSIsICAgICAgZmxvYXQobnAubWVhbihybF93YWdlcykpKQ0KICAgICAgICBzZWxmLmxvZ2dlci5yZWNvcmQoImNvbXAvd2FnZV9zcHJlYWQiLCAgICAgIGZsb2F0KG5wLm1heChybF93YWdlcykgLSBucC5taW4ocmxfd2FnZXMpKSkNCiAgICAgICAgc2VsZi5sb2dnZXIucmVjb3JkKCJjb21wL2F2Z19ybF93b3JrZXJzIiwgICBmbG9hdChucC5tZWFuKHJsX3dvcmtlcnMpKSkNCiAgICAgICAgc2VsZi5sb2dnZXIucmVjb3JkKCJjb21wL3dvcmtlcl9zcHJlYWQiLCAgICBmbG9hdChucC5tYXgocmxfd29ya2VycykgLSBucC5taW4ocmxfd29ya2VycykpKQ0KDQogICAgICAgIGVtcGxveWVkICAgID0gW3cgZm9yIHcgaW4gc2ltLndvcmtlcnMgaWYgdy5lbXBsb3llZF0NCiAgICAgICAgZW1wbG95X3JhdGUgPSBsZW4oZW1wbG95ZWQpIC8gbGVuKHNpbS53b3JrZXJzKSBpZiBzaW0ud29ya2VycyBlbHNlIDANCiAgICAgICAgYXZnX3Byb2ZpdCAgPSBmbG9hdChucC5tZWFuKFtmLnByb2ZpdCBmb3IgZiBpbiBzaW0uZmlybXNdKSkNCiAgICAgICAgbWFya2V0X3dhZ2UgPSBmbG9hdChucC5tZWFuKFtmLm1vbnRobHlfd2FnZSBmb3IgZiBpbiBzaW0uZmlybXNdKSkNCg0KICAgICAgICBzZWxmLmxvZ2dlci5yZWNvcmQoImVjb25vbXkvZW1wbG95bWVudF9yYXRlIiwgZW1wbG95X3JhdGUpDQogICAgICAgIHNlbGYubG9nZ2VyLnJlY29yZCgiZWNvbm9teS9hdmdfcHJvZml0X2FsbCIsICBhdmdfcHJvZml0KQ0KICAgICAgICBzZWxmLmxvZ2dlci5yZWNvcmQoImVjb25vbXkvbWFya2V0X2F2Z193YWdlIiwgbWFya2V0X3dhZ2UpDQoNCiAgICAgICAgaGV1cmlzdGljID0gW2YgZm9yIGYgaW4gc2ltLmZpcm1zIGlmIGYgbm90IGluIGVudi5ybF9maXJtc10NCiAgICAgICAgaWYgaGV1cmlzdGljOg0KICAgICAgICAgICAgaF9hdmcgPSBmbG9hdChucC5tZWFuKFtmLnByb2ZpdCBmb3IgZiBpbiBoZXVyaXN0aWNdKSkNCiAgICAgICAgICAgIHNlbGYubG9nZ2VyLnJlY29yZCgiY29tcC9ybF92c19oZXVyaXN0aWMiLCBmbG9hdChucC5tZWFuKHJsX3Byb2ZpdHMpKSAtIGhfYXZnKQ0KDQogICAgICAgIHRvdGFsID0gc2VsZi5hY3Rpb25fY291bnRzLnN1bSgpDQogICAgICAgIGlmIHRvdGFsID4gMDoNCiAgICAgICAgICAgIGZvciBpZHgsIG5hbWUgaW4gQUNUSU9OX05BTUVTLml0ZW1zKCk6DQogICAgICAgICAgICAgICAgc2VsZi5sb2dnZXIucmVjb3JkKGYiYWN0aW9ucy97bmFtZX0iLCBzZWxmLmFjdGlvbl9jb3VudHNbaWR4XSAvIHRvdGFsKQ0KDQogICAgICAgIHJldHVybiBUcnVlDQo=", "coop_train": "IyBjb29wZXJhdGl2ZS90cmFpbi5weSAg4oCUIHRyYWluIDMgUkwgZmlybXMgd2l0aCBzaGFyZWQgKGNvb3BlcmF0aXZlKSByZXdhcmQKIwojIFJlZm9ybWVkIHJ1bGVzOiBtYXJrZXQtcXVpdCwgT3B0aW9ucyAzKzQrNSwgc25hcCBhY3Rpb24gKDcpLCBlcXVhbF90ZXJtcwojIEh5cGVycGFyYW1zIGFsaWduZWQgd2l0aCByZWZvcm1lZC90cmFpbi5weToKIyAgIGVudF9jb2VmIDAuMDIgLT4gMC4wOCAgKGJyZWFrIGhvbGQtZG9taW5hbmNlIHBsYXRlYXUpCiMgICBuX3N0ZXBzICAxMDI0IC0+IDIwNDgKIyAgIHRvdGFsX3RpbWVzdGVwcyAyTSAtPiAzTSAgKGNvb3Agc2VlcyAzw5cgZmV3ZXIgbW9kZWwgc3RlcHMgcGVyIGd5bSBzdGVwKQoKZnJvbSBzYjNfY29udHJpYiBpbXBvcnQgTWFza2FibGVQUE8KZnJvbSBzYjNfY29udHJpYi5jb21tb24ud3JhcHBlcnMgaW1wb3J0IEFjdGlvbk1hc2tlcgpmcm9tIHN0YWJsZV9iYXNlbGluZXMzLmNvbW1vbi52ZWNfZW52IGltcG9ydCBEdW1teVZlY0VudiwgVmVjTm9ybWFsaXplCmZyb20gZmlybV9lbnYgaW1wb3J0IENvb3BGaXJtRW52CmZyb20gcmxfdmlzIGltcG9ydCBMYWJvck1ldHJpY3NDYWxsYmFjawoKCmRlZiBtYXNrX2ZuKGVudik6CiAgICByZXR1cm4gZW52LmFjdGlvbl9tYXNrcygpCgoKZGVmIGxpbmVhcl9zY2hlZHVsZShpbml0aWFsLCBmaW5hbD0xZS01KToKICAgIGRlZiBmbihwcm9ncmVzc19yZW1haW5pbmcpOgogICAgICAgIHJldHVybiBmaW5hbCArIHByb2dyZXNzX3JlbWFpbmluZyAqIChpbml0aWFsIC0gZmluYWwpCiAgICByZXR1cm4gZm4KCgpOX0VOVlMgPSA0CnJhd19lbnYgPSBEdW1teVZlY0VudihbCiAgICBsYW1iZGE6IEFjdGlvbk1hc2tlcihDb29wRmlybUVudigpLCBtYXNrX2ZuKQogICAgZm9yIF8gaW4gcmFuZ2UoTl9FTlZTKQpdKQoKZW52ID0gVmVjTm9ybWFsaXplKHJhd19lbnYsIG5vcm1fb2JzPUZhbHNlLCBub3JtX3Jld2FyZD1UcnVlLCBjbGlwX3Jld2FyZD01LjApCgptb2RlbCA9IE1hc2thYmxlUFBPKAogICAgIk1scFBvbGljeSIsCiAgICBlbnYsCiAgICB2ZXJib3NlPTEsCiAgICBuX3N0ZXBzPTIwNDgsCiAgICBiYXRjaF9zaXplPTI1NiwKICAgIG5fZXBvY2hzPTEwLAogICAgZ2FtbWE9MC45OSwKICAgIGdhZV9sYW1iZGE9MC45NSwKICAgIGxlYXJuaW5nX3JhdGU9bGluZWFyX3NjaGVkdWxlKDNlLTQsIDFlLTUpLAogICAgY2xpcF9yYW5nZT0wLjIsCiAgICBtYXhfZ3JhZF9ub3JtPTAuNSwKICAgIGVudF9jb2VmPTAuMDgsCiAgICB2Zl9jb2VmPTAuNSwKICAgIHBvbGljeV9rd2FyZ3M9ZGljdChuZXRfYXJjaD1bMjU2LCAyNTZdKSwKICAgIHRlbnNvcmJvYXJkX2xvZz0iLi90ZW5zb3Jib2FyZF9sb2dzLyIsCiAgICBkZXZpY2U9ImF1dG8iCikKCmNhbGxiYWNrID0gTGFib3JNZXRyaWNzQ2FsbGJhY2soCiAgICBsb2dfZGlyPSIuL3RlbnNvcmJvYXJkX2xvZ3MiLAogICAgYWxnb19uYW1lPSJDb29wX1JlZm9ybWVkX01hc2thYmxlUFBPIiwKICAgIGtlZXBfcnVucz0zLAopCgptb2RlbC5sZWFybigKICAgIHRvdGFsX3RpbWVzdGVwcz0zXzAwMF8wMDAsCiAgICBjYWxsYmFjaz1jYWxsYmFjawopCgptb2RlbC5zYXZlKCJjb29wX21vZGVsX2xvbmdydW4iKQplbnYuc2F2ZSgiY29vcF92ZWNub3JtX2xvbmdydW4ucGtsIikKcHJpbnQoIlNhdmVkOiBjb29wX21vZGVsX2xvbmdydW4uemlwICBjb29wX3ZlY25vcm1fbG9uZ3J1bi5wa2wiKQo=", "comp_train": "IyBjb21wZXRpdGl2ZS90cmFpbi5weSAg4oCUIHRyYWluIDMgUkwgZmlybXMgd2l0aCBjb21wZXRpdGl2ZSAocmVsYXRpdmUpIHJld2FyZAojCiMgUmVmb3JtZWQgcnVsZXM6IG1hcmtldC1xdWl0LCBPcHRpb25zIDMrNCs1LCBzbmFwIGFjdGlvbiAoNyksIGVxdWFsX3Rlcm1zCiMgSHlwZXJwYXJhbXMgYWxpZ25lZCB3aXRoIHJlZm9ybWVkL3RyYWluLnB5OgojICAgZW50X2NvZWYgMC4wMiAtPiAwLjA4ICAoYnJlYWsgaG9sZC1kb21pbmFuY2UgcGxhdGVhdSkKIyAgIG5fc3RlcHMgIDEwMjQgLT4gMjA0OAojICAgdG90YWxfdGltZXN0ZXBzIDJNIC0+IDNNCgpmcm9tIHNiM19jb250cmliIGltcG9ydCBNYXNrYWJsZVBQTwpmcm9tIHNiM19jb250cmliLmNvbW1vbi53cmFwcGVycyBpbXBvcnQgQWN0aW9uTWFza2VyCmZyb20gc3RhYmxlX2Jhc2VsaW5lczMuY29tbW9uLnZlY19lbnYgaW1wb3J0IER1bW15VmVjRW52LCBWZWNOb3JtYWxpemUKZnJvbSBmaXJtX2VudiBpbXBvcnQgQ29tcEZpcm1FbnYKZnJvbSBybF92aXMgaW1wb3J0IExhYm9yTWV0cmljc0NhbGxiYWNrCgoKZGVmIG1hc2tfZm4oZW52KToKICAgIHJldHVybiBlbnYuYWN0aW9uX21hc2tzKCkKCgpkZWYgbGluZWFyX3NjaGVkdWxlKGluaXRpYWwsIGZpbmFsPTFlLTUpOgogICAgZGVmIGZuKHByb2dyZXNzX3JlbWFpbmluZyk6CiAgICAgICAgcmV0dXJuIGZpbmFsICsgcHJvZ3Jlc3NfcmVtYWluaW5nICogKGluaXRpYWwgLSBmaW5hbCkKICAgIHJldHVybiBmbgoKCk5fRU5WUyA9IDQKcmF3X2VudiA9IER1bW15VmVjRW52KFsKICAgIGxhbWJkYTogQWN0aW9uTWFza2VyKENvbXBGaXJtRW52KCksIG1hc2tfZm4pCiAgICBmb3IgXyBpbiByYW5nZShOX0VOVlMpCl0pCgplbnYgPSBWZWNOb3JtYWxpemUocmF3X2Vudiwgbm9ybV9vYnM9RmFsc2UsIG5vcm1fcmV3YXJkPVRydWUsIGNsaXBfcmV3YXJkPTUuMCkKCm1vZGVsID0gTWFza2FibGVQUE8oCiAgICAiTWxwUG9saWN5IiwKICAgIGVudiwKICAgIHZlcmJvc2U9MSwKICAgIG5fc3RlcHM9MjA0OCwKICAgIGJhdGNoX3NpemU9MjU2LAogICAgbl9lcG9jaHM9MTAsCiAgICBnYW1tYT0wLjk5LAogICAgZ2FlX2xhbWJkYT0wLjk1LAogICAgbGVhcm5pbmdfcmF0ZT1saW5lYXJfc2NoZWR1bGUoM2UtNCwgMWUtNSksCiAgICBjbGlwX3JhbmdlPTAuMiwKICAgIG1heF9ncmFkX25vcm09MC41LAogICAgZW50X2NvZWY9MC4wOCwKICAgIHZmX2NvZWY9MC41LAogICAgcG9saWN5X2t3YXJncz1kaWN0KG5ldF9hcmNoPVsyNTYsIDI1Nl0pLAogICAgdGVuc29yYm9hcmRfbG9nPSIuL3RlbnNvcmJvYXJkX2xvZ3MvIiwKICAgIGRldmljZT0iYXV0byIKKQoKY2FsbGJhY2sgPSBMYWJvck1ldHJpY3NDYWxsYmFjaygKICAgIGxvZ19kaXI9Ii4vdGVuc29yYm9hcmRfbG9ncyIsCiAgICBhbGdvX25hbWU9IkNvbXBfUmVmb3JtZWRfTWFza2FibGVQUE8iLAogICAga2VlcF9ydW5zPTMsCikKCm1vZGVsLmxlYXJuKAogICAgdG90YWxfdGltZXN0ZXBzPTNfMDAwXzAwMCwKICAgIGNhbGxiYWNrPWNhbGxiYWNrCikKCm1vZGVsLnNhdmUoImNvbXBfbW9kZWxfbG9uZ3J1biIpCmVudi5zYXZlKCJjb21wX3ZlY25vcm1fbG9uZ3J1bi5wa2wiKQpwcmludCgiU2F2ZWQ6IGNvbXBfbW9kZWxfbG9uZ3J1bi56aXAgIGNvbXBfdmVjbm9ybV9sb25ncnVuLnBrbCIpCg=="}

FILE_MAP = {
    '/content/cooperative/model_rl.py': BLOBS['model_rl'],
    '/content/competitive/model_rl.py': BLOBS['model_rl'],
    '/content/cooperative/firm_env.py': BLOBS['coop_env'],
    '/content/competitive/firm_env.py': BLOBS['comp_env'],
    '/content/cooperative/rl_vis.py':   BLOBS['coop_vis'],
    '/content/competitive/rl_vis.py':   BLOBS['comp_vis'],
    '/content/cooperative/train.py':    BLOBS['coop_train'],
    '/content/competitive/train.py':    BLOBS['comp_train'],
}

for path, blob in FILE_MAP.items():
    pathlib.Path(path).write_bytes(base64.b64decode(blob))
    print(f'Wrote {path}')


## 2 - Train Cooperative

3 RL firms with **shared reward** (team average profit). ~1-2 hours.

In [ ]:
# Train Cooperative — streams live output
import subprocess, sys

proc = subprocess.Popen(
    [sys.executable, 'train.py'],
    cwd='/content/cooperative',
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
print('Return code:', proc.returncode)


## 3 - Train Competitive

3 RL firms with **individual reward minus 0.5 x peers** (relative profit). ~1-2 hours.

In [ ]:
# Train Competitive — streams live output
import subprocess, sys

proc = subprocess.Popen(
    [sys.executable, 'train.py'],
    cwd='/content/competitive',
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
print('
Return code:', proc.returncode)


## 4 - Download Results

In [ ]:
import zipfile, os
from google.colab import files

with zipfile.ZipFile('/content/trained_models.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in [
        '/content/cooperative/coop_model_longrun.zip',
        '/content/cooperative/coop_vecnorm_longrun.pkl',
        '/content/competitive/comp_model_longrun.zip',
        '/content/competitive/comp_vecnorm_longrun.pkl',
    ]:
        if os.path.exists(fname):
            zf.write(fname, os.path.basename(fname))
            print(f'Added {os.path.basename(fname)}')
        else:
            print(f'MISSING: {fname}')

files.download('/content/trained_models.zip')
